# SportSnap — Thí nghiệm Cải thiện ResNet50 (`ResNet50_v2`)

Notebook chuyên biệt để tối ưu mô hình **ResNet50** trên bộ dữ liệu 20 môn thể thao của SportSnap, khắc phục các hạn chế đã phát hiện ở bản `ResNet50_finetune_best.keras` (v1):

1. **Tách `valid` nội bộ đủ lớn (`12%` từ `train`, ~720 ảnh phân tầng)** để `EarlyStopping` & `ModelCheckpoint` chọn điểm dừng ổn định thay vì dựa vào `valid/` gốc chỉ có 100 ảnh; giữ nguyên `valid/` (100) và `test/` (100) làm tập đánh giá khách quan.
2. **Thu hẹp tầng mở khóa khi fine-tune (`UNFREEZE_LAYERS = 32` — khối `conv5_block*`)** thay vì `120` lớp (tránh làm trôi đặc trưng trung gian `conv3/conv4` trên tập nhỏ 6.000 ảnh, giảm nhầm `football` ↔ `baseball`/`hockey`), đồng thời **đóng băng toàn bộ `BatchNormalization`**.
3. **Optimizer & Regularization tốt hơn:** `AdamW` (`weight_decay=1e-4`, `clipnorm=1.0`), `Dropout(0.4)`, `LABEL_SMOOTHING=0.05`, Cosine LR decay có warm-up.
4. **Tăng cường dữ liệu sửa lỗi thang đo + MixUp nhẹ (`MIXUP_ALPHA=0.2`):** sửa `GaussianNoise` từ `0.05` lên `6.0` (phù hợp thang pixel `[0, 255]`), thêm `RandomZoom`/dịch chuyển/lật/ánh sáng + MixUp để giảm phụ thuộc nền (background bias) và cải thiện ảnh ngoài (`anh-ngoai`).
5. **Test-Time Augmentation (TTA — lật ngang + zoom trung tâm)** và **đối chiếu trực tiếp `ResNet50 v1` vs `ResNet50 v2` vs `ResNet50 v2 + TTA`** kèm **Grad-CAM (`conv5_block3_out`)**.

> **An toàn dự án:** Notebook này chỉ xuất mô hình thử nghiệm vào `OUT_DIR` (`./resnet50_v2_outputs` hoặc `/kaggle/working/resnet50_v2_outputs`), **không** ghi đè `backend/models/` hay `models/` gốc.


## 1. Import thư viện & cấu hình tham số


In [ ]:
import os, glob, json, math, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image, ImageOps

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ==== THAM SỐ HUẤN LUYỆN RESNET50 V2 ====
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_FROM_TRAIN = 0.12        # tách 12% từ train (~720 ảnh) làm val nội bộ để tránh overfit valid/ 100 ảnh

EPOCHS_HEAD = 10             # giai đoạn 1: train head (đóng băng toàn bộ backbone)
EPOCHS_FINETUNE = 25         # giai đoạn 2: fine-tune khối conv5_block*
HEAD_LR = 1e-3
FINETUNE_LR = 2e-5
WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.4
LABEL_SMOOTHING = 0.05
UNFREEZE_LAYERS = 32         # 32 lớp cuối = toàn bộ khối conv5_block* (thay vì 120 lớp ở v1)

USE_MIXUP = True
MIXUP_ALPHA = 0.2

RUN_TRAIN = True             # True: huấn luyện ResNet50_v2; False: chỉ nạp ResNet50_v2_best.keras nếu đã có
MAX_STEPS_SMOKE = None       # Đặt số nhỏ (vd 2) nếu chỉ muốn test nhanh luồng chạy, None = chạy toàn bộ

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 2. Dò đường dẫn dữ liệu, `labels.json` & checkpoint v1 đối chiếu


In [ ]:
def auto_find_data_root():
    if os.path.exists("/kaggle/input"):
        for root, dirs, _ in os.walk("/kaggle/input"):
            if "train" in dirs and "valid" in dirs:
                return root
    for p in [
        r"d:\tgmt\datasets\sports-classification",
        "../datasets/sports-classification",
        "./datasets/sports-classification",
        "/kaggle/input/datasets/kienmaitrung287/data-sportsnap",
    ]:
        if os.path.exists(p):
            return os.path.abspath(p)
    raise FileNotFoundError("Không tìm thấy thư mục sports-classification (train/valid/test).")

def auto_find_ext_dir():
    if os.path.exists("/kaggle/input"):
        for root, dirs, _ in os.walk("/kaggle/input"):
            if "anh-ngoai" in dirs:
                return os.path.join(root, "anh-ngoai")
    for p in [
        r"d:\tgmt\datasets\anh-ngoai",
        "../datasets/anh-ngoai",
        "./datasets/anh-ngoai",
    ]:
        if os.path.exists(p):
            return os.path.abspath(p)
    return None

def find_file(filename):
    for d in ["./models", "../models", r"d:\tgmt\models", "/kaggle/working", "."]:
        p = os.path.join(d, filename)
        if os.path.exists(p):
            return os.path.abspath(p)
    if os.path.exists("/kaggle/input"):
        hits = sorted(glob.glob(f"/kaggle/input/**/{filename}", recursive=True))
        if hits:
            return hits[0]
    return None

DATA_ROOT = auto_find_data_root()
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VALID_DIR = os.path.join(DATA_ROOT, "valid")
TEST_DIR  = os.path.join(DATA_ROOT, "test")
EXT_DIR   = auto_find_ext_dir()

OUT_DIR = "/kaggle/working/resnet50_v2_outputs" if os.path.exists("/kaggle/working") else os.path.abspath("./resnet50_v2_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

labels_file = find_file("labels.json")
if labels_file:
    with open(labels_file, "r", encoding="utf-8") as f:
        CLASS_NAMES = json.load(f)
else:
    CLASS_NAMES = sorted(d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d)))[:20]

NUM_CLASSES = len(CLASS_NAMES)
V1_CKPT = find_file("ResNet50_finetune_best.keras")
V2_CKPT = os.path.join(OUT_DIR, "ResNet50_v2_best.keras")

print("DATA_ROOT :", DATA_ROOT)
print("EXT_DIR   :", EXT_DIR)
print("OUT_DIR   :", OUT_DIR)
print("V1_CKPT   :", V1_CKPT)
print("NUM_CLASSES:", NUM_CLASSES, CLASS_NAMES[:5], "...")


## 3. Chia tập train/val nội bộ (phân tầng) & Pipeline Augmentation v2 + MixUp


In [ ]:
VALID_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def collect_split(directory, class_names):
    paths, labels = [], []
    for idx, cname in enumerate(class_names):
        cdir = os.path.join(directory, cname)
        if not os.path.isdir(cdir):
            continue
        for fn in sorted(os.listdir(cdir)):
            if fn.lower().endswith(VALID_EXTS):
                paths.append(os.path.join(cdir, fn))
                labels.append(idx)
    return np.array(paths), np.array(labels, dtype=np.int32)

tr_paths_all, tr_labels_all = collect_split(TRAIN_DIR, CLASS_NAMES)
val_orig_paths, val_orig_labels = collect_split(VALID_DIR, CLASS_NAMES)
test_paths, test_labels = collect_split(TEST_DIR, CLASS_NAMES)

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FROM_TRAIN, random_state=SEED)
tr_idx, val_in_idx = next(sss.split(tr_paths_all, tr_labels_all))
tr_in_paths, tr_in_labels = tr_paths_all[tr_idx], tr_labels_all[tr_idx]
val_in_paths, val_in_labels = tr_paths_all[val_in_idx], tr_labels_all[val_in_idx]

print(f"Train nội bộ : {len(tr_in_paths)} ảnh")
print(f"Val nội bộ   : {len(val_in_paths)} ảnh (12% từ train, phân tầng đều {NUM_CLASSES} lớp)")
print(f"Valid gốc    : {len(val_orig_paths)} ảnh | Test gốc: {len(test_paths)} ảnh")

# Bộ tăng cường dữ liệu v2: sửa thang đo nhiễu Gaussian (stddev=6.0 trên [0, 255],
# vì keras.layers.GaussianNoise chỉ nhận stddev trong [0, 1])
# và giữ biến đổi hình học vừa phải để không phá bối cảnh dụng cụ/sân bãi.
data_augmentation_v2 = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.12),
    layers.RandomZoom(height_factor=(-0.18, 0.15)),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.25),
    layers.RandomBrightness(0.25, value_range=(0, 255)),
], name="data_augmentation_v2")

def apply_aug_and_noise(x):
    x = data_augmentation_v2(x, training=True)
    return x + tf.random.normal(tf.shape(x), mean=0.0, stddev=6.0)

def decode_resize(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE, method="bilinear", antialias=True)
    return tf.cast(img, tf.float32), label

def mixup_batch(images, onehot_labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    g1 = tf.random.gamma([batch_size, 1], alpha=alpha)
    g2 = tf.random.gamma([batch_size, 1], alpha=alpha)
    lam = g1 / (g1 + g2 + 1e-8)
    lam_img = tf.reshape(lam, [batch_size, 1, 1, 1])
    perm = tf.random.shuffle(tf.range(batch_size))
    mixed_x = lam_img * images + (1.0 - lam_img) * tf.gather(images, perm)
    mixed_y = lam * onehot_labels + (1.0 - lam) * tf.gather(onehot_labels, perm)
    return mixed_x, mixed_y

AUTOTUNE = tf.data.AUTOTUNE

def make_resnet_ds(paths, labels, augment=False, onehot=True, use_mixup=False, cache=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    if augment:
        ds = ds.shuffle(min(len(paths), 2000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if onehot:
        ds = ds.map(lambda x, y: (x, tf.one_hot(y, NUM_CLASSES)), num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(lambda x, y: (apply_aug_and_noise(x), y), num_parallel_calls=AUTOTUNE)
        if use_mixup:
            ds = ds.map(lambda x, y: mixup_batch(x, y, MIXUP_ALPHA), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (resnet_preprocess(tf.clip_by_value(x, 0.0, 255.0)), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds    = make_resnet_ds(tr_in_paths, tr_in_labels, augment=True,  onehot=True,  use_mixup=USE_MIXUP, cache=True)
val_in_ds   = make_resnet_ds(val_in_paths, val_in_labels, augment=False, onehot=True, cache=True)
val_orig_ds = make_resnet_ds(val_orig_paths, val_orig_labels, augment=False, onehot=False, cache=False)
test_ds     = make_resnet_ds(test_paths, test_labels, augment=False, onehot=False, cache=False)


## 4. Xây dựng & huấn luyện `ResNet50_v2` (2 giai đoạn: Head → Fine-tune `conv5_block*`)


In [ ]:
def build_resnet50_v2(num_classes):
    base = ResNet50(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="ResNet50_v2")
    return model, base

def freeze_batchnorm(base_model):
    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

def unfreeze_top_layers(base_model, n_layers):
    base_model.trainable = True
    for layer in base_model.layers[:-n_layers]:
        layer.trainable = False
    freeze_batchnorm(base_model)

def cosine_schedule(peak_lr, total_epochs, min_lr=1e-7, warmup_epochs=2):
    def fn(epoch):
        if epoch < warmup_epochs:
            return peak_lr * (epoch + 1) / (warmup_epochs + 1)
        t = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return min_lr + 0.5 * (peak_lr - min_lr) * (1.0 + math.cos(math.pi * min(1.0, t)))
    return fn

def make_loss():
    return keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

if RUN_TRAIN:
    model_v2, base_v2 = build_resnet50_v2(NUM_CLASSES)

    tr_fit = train_ds.take(MAX_STEPS_SMOKE) if MAX_STEPS_SMOKE else train_ds
    va_fit = val_in_ds.take(MAX_STEPS_SMOKE) if MAX_STEPS_SMOKE else val_in_ds

    # ---- Giai đoạn 1: Warm-up classification head ----
    model_v2.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=HEAD_LR, weight_decay=WEIGHT_DECAY, clipnorm=1.0),
        loss=make_loss(),
        metrics=["accuracy"],
    )
    h1 = model_v2.fit(
        tr_fit,
        validation_data=va_fit,
        epochs=EPOCHS_HEAD,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
        ],
    )

    # ---- Giai đoạn 2: Fine-tune khối conv5_block* (32 lớp cuối, giữ BN đóng băng) ----
    unfreeze_top_layers(base_v2, UNFREEZE_LAYERS)
    print(
        f"Trainable trong ResNet50 base: "
        f"{sum(1 for l in base_v2.layers if l.trainable)} / {len(base_v2.layers)} lớp "
        f"(BN trainable={sum(1 for l in base_v2.layers if l.trainable and isinstance(l, layers.BatchNormalization))})"
    )

    model_v2.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=FINETUNE_LR, weight_decay=WEIGHT_DECAY, clipnorm=1.0),
        loss=make_loss(),
        metrics=["accuracy"],
    )
    h2 = model_v2.fit(
        tr_fit,
        validation_data=va_fit,
        epochs=EPOCHS_FINETUNE,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(V2_CKPT, monitor="val_loss", save_best_only=True),
            keras.callbacks.LearningRateScheduler(cosine_schedule(FINETUNE_LR, EPOCHS_FINETUNE)),
        ],
    )
    if not os.path.exists(V2_CKPT):
        model_v2.save(V2_CKPT)
    else:
        model_v2 = keras.models.load_model(V2_CKPT, compile=False)

    with open(os.path.join(OUT_DIR, "labels.json"), "w", encoding="utf-8") as f:
        json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)
    print("Đã lưu ResNet50_v2 tại:", V2_CKPT)
else:
    assert os.path.exists(V2_CKPT), f"Chưa có {V2_CKPT} — hãy đặt RUN_TRAIN = True để huấn luyện."
    model_v2 = keras.models.load_model(V2_CKPT, compile=False)
    print("Đã nạp ResNet50_v2 từ:", V2_CKPT)


## 5. Suy luận chuẩn & TTA (Test-Time Augmentation), đối chiếu `v1` vs `v2`


In [ ]:
def load_images_uint8(paths):
    '''Đọc ảnh theo đúng hợp đồng SPEC.md muc 4 (EXIF transpose, RGB, resize BILINEAR 224x224).'''
    arr = np.empty((len(paths),) + IMG_SIZE + (3,), dtype=np.float32)
    for i, p in enumerate(paths):
        with Image.open(p) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            if im.size != IMG_SIZE:
                im = im.resize(IMG_SIZE, Image.BILINEAR)
            arr[i] = np.asarray(im, dtype=np.float32)
    return arr

def predict_single(model, imgs_f32):
    x = resnet_preprocess(imgs_f32.copy())
    return model.predict(x, batch_size=BATCH_SIZE, verbose=0)

def predict_tta(model, imgs_f32):
    '''TTA: trung bình xác suất của ảnh gốc + lật ngang + zoom trung tâm 88%.'''
    p_orig = predict_single(model, imgs_f32)
    p_flip = predict_single(model, imgs_f32[:, :, ::-1, :])

    h, w = IMG_SIZE
    ch, cw = int(h * 0.88), int(w * 0.88)
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    cropped = tf.image.resize(imgs_f32[:, y0:y0+ch, x0:x0+cw, :], IMG_SIZE, method="bilinear", antialias=True).numpy()
    p_crop = predict_single(model, cropped)
    return (0.50 * p_orig + 0.30 * p_flip + 0.20 * p_crop)

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    d = 1.0 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - h, c + h

def eval_probs(probs, y_true):
    y_pred = probs.argmax(1)
    top5 = np.argsort(-probs, axis=1)[:, :5]
    top1_hits = int((y_pred == y_true).sum())
    top5_hits = int(sum(y_true[i] in top5[i] for i in range(len(y_true))))
    n = len(y_true)
    return {
        "n": n,
        "top1": top1_hits / n,
        "top1_hits": top1_hits,
        "top5": top5_hits / n,
        "top5_hits": top5_hits,
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "ci95": wilson_ci(top1_hits, n),
    }

val_imgs  = load_images_uint8(val_orig_paths)
test_imgs = load_images_uint8(test_paths)

eval_models = []
if V1_CKPT and os.path.exists(V1_CKPT):
    model_v1 = keras.models.load_model(V1_CKPT, compile=False)
    eval_models.append(("ResNet50_v1 (Single)", model_v1, False))
    eval_models.append(("ResNet50_v1 + TTA",    model_v1, True))

eval_models.append(("ResNet50_v2 (Single)", model_v2, False))
eval_models.append(("ResNet50_v2 + TTA",    model_v2, True))

ext_imgs = ext_labels = None
if EXT_DIR and os.path.exists(EXT_DIR):
    ext_paths, ext_labels = collect_split(EXT_DIR, CLASS_NAMES)
    if len(ext_paths) > 0:
        ext_imgs = load_images_uint8(ext_paths)

rows = []
stored_probs = {}
for label_name, m, use_tta in eval_models:
    fn = predict_tta if use_tta else predict_single
    p_val  = fn(m, val_imgs)
    p_test = fn(m, test_imgs)
    r_val  = eval_probs(p_val, val_orig_labels)
    r_test = eval_probs(p_test, test_labels)
    row = {
        "Mô hình": label_name,
        "Valid Acc": round(r_val["top1"], 4),
        "Valid F1":  round(r_val["macro_f1"], 4),
        "Test Acc":  round(r_test["top1"], 4),
        "Test F1":   round(r_test["macro_f1"], 4),
    }
    if ext_imgs is not None:
        p_ext = fn(m, ext_imgs)
        r_ext = eval_probs(p_ext, ext_labels)
        row["Ngoài Top-1"] = f"{r_ext['top1']:.3f} ({r_ext['top1_hits']}/{r_ext['n']})"
        row["Ngoài Top-5"] = f"{r_ext['top5']:.3f} ({r_ext['top5_hits']}/{r_ext['n']})"
    rows.append(row)
    stored_probs[label_name] = (p_val, p_test)

cmp_df = pd.DataFrame(rows)
print(cmp_df.to_string(index=False))
cmp_df.to_csv(os.path.join(OUT_DIR, "comparison.csv"), index=False)


## 6. Kiểm tra riêng các ca `valid` bị nhầm trước đó & Grad-CAM (`conv5_block3_out`)


In [ ]:
# Kiểm tra lỗi trên valid của ResNet50_v2 (Single & TTA)
for tag in ["ResNet50_v2 (Single)", "ResNet50_v2 + TTA"]:
    p_val, _ = stored_probs[tag]
    preds = p_val.argmax(1)
    wrong = np.where(preds != val_orig_labels)[0]
    print(f"=== {tag}: {len(wrong)}/{len(val_orig_labels)} ảnh sai trên valid ===")
    for idx in wrong:
        rel = os.path.relpath(val_orig_paths[idx], VALID_DIR)
        print(f"  {rel}: thực={CLASS_NAMES[val_orig_labels[idx]]} -> đoán={CLASS_NAMES[preds[idx]]} ({p_val[idx, preds[idx]]:.3f})")

def make_resnet_gradcam(img_f32, model, layer_name="conv5_block3_out"):
    base = next(l for l in model.layers if hasattr(l, "layers"))
    target = base.get_layer(layer_name)
    conv_model = keras.Model(base.input, target.output)

    cls_in = keras.Input(shape=conv_model.output.shape[1:])
    y = cls_in
    for l in model.layers:
        if l is not model.layers[0] and l is not base:
            y = l(y)
    cls_model = keras.Model(cls_in, y)

    x_in = resnet_preprocess(img_f32[None].copy())
    with tf.GradientTape() as tape:
        conv_out = conv_model(x_in)
        tape.watch(conv_out)
        preds = cls_model(conv_out)
        pred_idx = tf.argmax(preds[0])
        score = preds[:, pred_idx]
    grads = tape.gradient(score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    hm = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
    hm = tf.maximum(hm, 0) / (tf.math.reduce_max(hm) + 1e-8)
    return hm.numpy(), int(pred_idx)

# Hiển thị Grad-CAM cho 3 ảnh football trong valid (vốn bị v1 đoán nhầm) + 3 ảnh ngẫu nhiên từ test
check_indices = [i for i, p in enumerate(val_orig_paths) if "football" in p][:3]
if check_indices:
    plt.figure(figsize=(12, 4))
    for k, idx in enumerate(check_indices):
        hm, p_idx = make_resnet_gradcam(val_imgs[idx], model_v2)
        hm_u8 = np.uint8(255 * cv2.resize(hm, IMG_SIZE))
        hm_rgb = cv2.cvtColor(cv2.applyColorMap(hm_u8, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        ov = np.uint8(val_imgs[idx] * 0.6 + hm_rgb * 0.4)
        plt.subplot(1, len(check_indices), k + 1)
        plt.imshow(ov)
        plt.axis("off")
        ok = (p_idx == val_orig_labels[idx])
        plt.title(f"Thực: {CLASS_NAMES[val_orig_labels[idx]]}\nĐoán: {CLASS_NAMES[p_idx]}",
                  color="green" if ok else "red", fontsize=10)
    plt.suptitle("Grad-CAM ResNet50_v2 trên các ảnh valid/football")
    plt.tight_layout()
    plt.show()
